<a href="https://colab.research.google.com/github/cgalla4/churn-risk-assistant/blob/main/churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Risk Assistant
Predicts which telecom customers are likely to leave, then uses generative AI to write next-step recommendations for account managers.

## 1. Business Problem & Data
About 26.5% of customers churned. The goal is to identify at-risk customers early so a retention team can reach out.

In [16]:
# Load IBM's public Telco Customer Churn dataset (7,043 customers) directly from IBM's GitHub
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Check column types to identify any data stored in the wrong format
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Baseline: 26.5% churned, so a model that predicts "no one leaves" would still be 73.5% accurate
df['Churn'].value_counts(normalize=True)

,proportion
Churn,
No,0.73463
Yes,0.26537


## 2. Data Cleaning
TotalCharges was stored as text because 11 brand-new customers (tenure = 0) had blank values. I filled them with $0, their true value.

In [5]:
# Find the blank TotalCharges values and see what those customers have in common
df[df["TotalCharges"].str.strip() == ""][["tenure","MonthlyCharges","TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,
3331,0,19.85,
3826,0,25.35,
4380,0,20.00,
5218,0,19.70,
6670,0,73.35,


In [6]:
# All blank rows are brand-new customers (tenure = 0) who haven't been billed yet, so $0 is the true value
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 3. Preparing the Data
Converted text columns to 0/1 columns and split the data 80/20 into training and test sets.

In [7]:
# customerID is just a label, so drop it
# Turn Churn into 1/0 and text columns into 0/1 columns so the models can read them
df_model = df.drop(columns=["customerID"])
df_model["Churn"] = (df_model["Churn"] == "Yes").astype(int)
df_model = pd.get_dummies(df_model, drop_first=True)
print(df_model.shape)

(7043, 31)


In [8]:
# Hold back 20% of customers to test the model on data it hasn't seen
# stratify keeps the same 26.5% churn rate in both groups
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(5634, 30) (1409, 30)


## 4. Modeling
Compared logistic regression and a random forest.
Logistic regression performed better, and I balanced it to catch more churners. The balanced model catches 79% of churners (AUC 0.84). It flags more false alarms, but a check-in call costs far less than losing a customer.

In [9]:
# Start with logistic regression: simple, fast, and easy to explain to business users
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_model = LogisticRegression(max_iter=5000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, log_pred))
print("AUC:", round(roc_auc_score(y_test, log_prob),3))

              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

AUC: 0.842


In [10]:
# Compare against a random forest to see if a more complex model does better
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, rf_pred))
print("AUC:", round(roc_auc_score(y_test, rf_prob),3))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.64      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

AUC: 0.827


Logistic regression won on recall and AUC, so the simpler model is the better choice here.

In [11]:
# Churners are only ~26% of customers, so weight them more heavily to catch more of them
# Trade-off: a false alarm costs a phone call, but a missed churner costs the whole customer
bal_model = LogisticRegression(max_iter=5000, class_weight="balanced")
bal_model.fit(X_train, y_train)

bal_pred = bal_model.predict(X_test)
bal_prob = bal_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, bal_pred))
print("AUC:", round(roc_auc_score(y_test, bal_prob), 3))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

AUC: 0.842


## 5. What Drives Churn
Fiber internet and electronic check payments raise churn risk the most. 1–2 year contracts, online security and tech support lower it. These are patterns, not proof of cause.


In [12]:
# Positive weight = raises churn risk, negative = lowers it
coefs = pd.Series(bal_model.coef_[0], index=X.columns).sort_values()

print("Biggest churn risks: ")
print(coefs.tail(8))
print("\nBiggest churn protectors: ")
print(coefs.head(8))

Biggest churn risks: 
PaperlessBilling_Yes              0.338062
MultipleLines_Yes                 0.352646
PhoneService_Yes                  0.400636
PaymentMethod_Electronic check    0.405387
StreamingTV_Yes                   0.446693
MultipleLines_No phone service    0.461791
StreamingMovies_Yes               0.472457
InternetService_Fiber optic       1.348392
dtype: float64

Biggest churn protectors: 
Contract_Two year                    -1.411227
Contract_One year                    -0.716929
OnlineSecurity_Yes                   -0.307961
TechSupport_Yes                      -0.241962
Dependents_Yes                       -0.228024
OnlineBackup_No internet service     -0.201742
InternetService_No                   -0.201742
OnlineSecurity_No internet service   -0.201742
dtype: float64


## 6. AI Retention Recommendations

In [13]:
# Show the 10 customers most likely to leave
risk_list = df.loc[X_test.index, ["customerID", "tenure", "Contract", "InternetService",
                                  "PaymentMethod", "MonthlyCharges", "TechSupport"]].copy()
risk_list["churn_prob"] = bal_prob.round(2)

top10 = risk_list.sort_values("churn_prob", ascending=False).head(10)
top10

,customerID,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,TechSupport,churn_prob
3380,5178-LMXOP,1,Month-to-month,Fiber optic,Electronic check,95.10,No,0.94
6866,0295-PPHDO,1,Month-to-month,Fiber optic,Electronic check,95.45,No,0.93
2631,6861-XWTWQ,7,Month-to-month,Fiber optic,Electronic check,99.25,No,0.93
4039,8161-QYMTT,7,Month-to-month,Fiber optic,Electronic check,94.10,No,0.92
3346,2545-EBUPK,2,Month-to-month,Fiber optic,Electronic check,84.05,No,0.92
6894,1400-MMYXY,3,Month-to-month,Fiber optic,Electronic check,105.90,No,0.92
4585,1069-XAIEM,1,Month-to-month,Fiber optic,Electronic check,85.05,No,0.92
3727,9057-SIHCH,3,Month-to-month,Fiber optic,Electronic check,96.60,No,0.92
2797,6023-YEBUP,3,Month-to-month,Fiber optic,Electronic check,100.95,No,0.92
6365,8884-ADFVN,7,Month-to-month,Fiber optic,Electronic check,101.95,No,0.92


In [22]:
# Load the key from Colab Secrets so it never appears in the code
# Give up on any request after 60 seconds so one slow request can't freeze the notebook
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 60000})
MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]  # lighter models are less overloaded on the free tier

In [23]:
import json, os, time
pd.set_option("display.max_colwidth", None)  # show full notes instead of cutting them off

# Give the AI a fixed list of offers so it can't invent prices or discounts
OFFERS = "1-year contract with 10% off; free tech support for 6 months; $5/month off for switching to autopay"

def build_prompt(customers):
    # Describe all 10 customers in one prompt so only one AI request is needed
    lines = []
    for _, r in customers.iterrows():
        lines.append(f"{r.customerID}: {r.churn_prob:.0%} churn risk, {r.tenure} months as a customer, "
                     f"{r.Contract} contract, {r.InternetService} internet, pays by {r.PaymentMethod}, "
                     f"${r.MonthlyCharges}/month, tech support: {r.TechSupport}")
    customer_text = "\n".join(lines)
    return f"""You are helping a telecom account manager keep customers from leaving.
Our model found that fiber internet and electronic check payments raise churn risk,
and 1-2 year contracts and tech support lower it.
Only recommend offers from this list: {OFFERS}. Do not invent any other prices or discounts.

Customers:
{customer_text}

Return JSON: a list with one object per customer, with the keys
"customerID", "why_at_risk", "offer", "opening_line". Keep each value to one sentence."""

def ask_gemini(prompt, tries=3):
    # Try each model in turn, retrying if the AI service is busy
    for attempt in range(tries):
        for model in MODELS:
            try:
                response = client.models.generate_content(
                    model=model, contents=prompt,
                    config={"response_mime_type": "application/json"})  # ask for structured data, not free text
                print(f"Notes written by {model}")
                return json.loads(response.text)
            except Exception as e:
                print(f"{model} attempt {attempt + 1} failed: {str(e)[:80]}")
        time.sleep(15 * (attempt + 1))
    return None

NOTES_FILE = "ai_notes.csv"
if os.path.exists(NOTES_FILE):
    # Reuse saved notes so re-running the notebook doesn't spend AI quota
    notes_df = pd.read_csv(NOTES_FILE)
    print("Loaded saved AI notes.")
else:
    result = ask_gemini(build_prompt(top10))
    if result is None:
        raise RuntimeError("AI service unavailable - try again later.")
    notes_df = pd.DataFrame(result)
    notes_df.to_csv(NOTES_FILE, index=False)
    print("Generated and saved AI notes.")

# Join the AI notes to the at-risk list
final = top10.merge(notes_df, on="customerID")
final[["customerID", "churn_prob", "why_at_risk", "offer", "opening_line"]]

Notes written by gemini-3.5-flash-lite
Generated and saved AI notes.


,customerID,churn_prob,why_at_risk,offer,opening_line
0,5178-LMXOP,0.94,"The customer is at high risk due to having fiber internet, an electronic check payment method, and a month-to-month contract without tech support.",1-year contract with 10% off,We noticed you are on a month-to-month plan and want to help you save on your fiber internet.
1,0295-PPHDO,0.93,"The customer faces a high churn risk because they use fiber internet, pay by electronic check, and lack a long-term contract or tech support.",free tech support for 6 months,We would love to help you get the most out of your fiber internet with some complimentary assistance.
2,6861-XWTWQ,0.93,"High churn risk is driven by fiber internet, electronic check payments, and a lack of tech support on a short-term contract.",$5/month off for switching to autopay,We can help lower your monthly bill by updating how you pay for your service.
3,8161-QYMTT,0.92,"The combination of fiber internet, electronic check payment, and a month-to-month agreement puts this customer at significant risk.",1-year contract with 10% off,Let us reward your loyalty with a discounted annual rate for your internet service.
4,2545-EBUPK,0.92,"This customer is at risk due to a month-to-month fiber internet plan, electronic check billing, and no tech support.",free tech support for 6 months,We want to ensure your fiber internet experience is seamless and fully supported.
5,1400-MMYXY,0.92,"High churn probability stems from fiber internet, electronic check billing, and the absence of a contract or tech support.",$5/month off for switching to autopay,Switching your payment method is a quick way to reduce your monthly expenses with us.
6,1069-XAIEM,0.92,"The user is at risk because they have fiber internet, pay via electronic check, and are on a flexible month-to-month plan.",1-year contract with 10% off,Lock in your service with a yearly plan and enjoy significant savings on your bill.
7,9057-SIHCH,0.92,"High churn risk is caused by fiber optic service, electronic check payments, and a month-to-month commitment.",free tech support for 6 months,We want to give you extra peace of mind with complimentary expert support for your connection.
8,6023-YEBUP,0.92,"The customer is likely to leave due to high monthly costs on fiber internet, electronic check billing, and no contract.",$5/month off for switching to autopay,We can make your billing more convenient while putting money back in your pocket.
9,8884-ADFVN,0.92,"High churn is linked to fiber internet, electronic check payments, and a lack of long-term contract stability.",1-year contract with 10% off,Secure a lower rate for your fiber internet by transitioning to an annual agreement.
